In [ ]:
import polars as pl
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import math
import itertools
from procompa import get_project_root, get_data_dir

PRJ_ROOT = get_project_root()
data_dir = PRJ_ROOT / "data"
data_dir_YM = get_data_dir() / "26.03_yeast.MAP"

## Understand pooled data

In [ ]:
import af3io, pooled_ppi

In [ ]:
pp = pooled_ppi.PooledPredictionsDb("/cluster/work/beltrao/jjaenes/25.12_pooled-ppi-yeast/data-26.04")
pairs_ = pp.pairs.query('uniprot_id1 in @proteins & uniprot_id2 in @proteins').sort_values('af3_pair').reset_index(drop=True)


Select High confidence Score

In [ ]:
YM_COMPLEX_overlap = pl.read_csv(data_dir /"Complex_Portal/YeastMap/complex_db_complete_unique_overlap_complex_yeastmap_by_complex.csv")
complex_db_entries = pl.read_csv(data_dir / "Complex_Portal/Saccharomyces_cerevisiae_ComplexTab.tsv", separator= "\t" )

In [ ]:
#firstly focus on predition taht have an extremly high confidence score of 1
#high_confidence_predictions = YM_COMPLEX_overlap.filter(pl.col("ComplexConfidence") == 1)

#add stoichiometry information from complex database (if given)
YM_COMPLEX_overlap = (
    YM_COMPLEX_overlap.join(
        complex_db_entries.select([
            "#Complex ac", 
            "Identifiers (and stoichiometry) of molecules in complex"
        ]),
        on="#Complex ac",
        how="left"
    )
    .with_columns(
        stoichiometry_known = ~pl.col("Identifiers (and stoichiometry) of molecules in complex").str.contains(r"\(0\)"),
        solely_proteins = ~pl.col("Identifiers (and stoichiometry) of molecules in complex").str.contains(r":")
    )
)

for which homdimers do i have the pdb file

In [ ]:
homodimer_with_pdb = pl.read_csv(data_dir/ "Pipeline/merged_pdbs/pdb_file_names.csv")

In [ ]:
homodimer_set = homodimer_with_pdb["uniprot_id_homodimer"].to_list()

YM_COMPLEX_overlap = YM_COMPLEX_overlap.with_columns(
    pl.col("true_complex").str.split(" ").alias("_proteins")
).with_columns(
    pl.col("_proteins")
      .list.eval(pl.element().is_in(homodimer_set))
      .list.sum()
      .alias("n_proteins_with_homodimer_pdb"),
    pl.col("_proteins")
      .list.eval(pl.element().is_in(homodimer_set))
      .list.all()
      .alias("all_proteins_have_pdb"),
).drop("_proteins")

strict pipeline dataset for complexes where all proteins have homodimers on Alfafold database

In [ ]:
pdb_pipeline_dataset = YM_COMPLEX_overlap.filter((pl.col("solely_proteins") == True)& (pl.col("stoichiometry_known") == True)& pl.col("all_proteins_have_pdb")== True)

In [ ]:
pdb_pipeline_dataset.write_csv(data_dir/ "Pipeline/pdb_present_first_setup_pipeline_complexes.csv")

initial first subset for the pipeline

In [ ]:
pipeline_complexes = high_confidence_predictions.filter((pl.col("solely_proteins") == True)& (pl.col("stoichiometry_known") == True)& (pl.col("jaccard_similarity") == 1))

In [ ]:
pipeline_complexes.write_csv(data_dir/ "Pipeline/first_setup_pipeline_complexes.csv")

In [ ]:
uniprot_id_complexes_pipeline = pipeline_complexes.get_column("predicted_complex").str.split(" " ).explode().unique().to_list()

#pl.DataFrame({"uniprot_id": uniprot_id_complexes_pipeline}).write_csv(data_dir /"Pipeline/uniprot_ids_first_setup.csv")

## cretae subset for combfold where i have pdb files for all proteins in the complex to understand when it assembles and wen it doenst

In [ ]:
YM_COMPLEX_overlap_homomerdimer_pdb = YM_COMPLEX_overlap.filter((pl.col("all_proteins_have_pdb") == True) &(pl.col("stoichiometry_known")))#YM_COMPLEX_overlap based on homodimers for whcih i have pdb and cell beforehand

In [ ]:
YM_COMPLEX_overlap_homomerdimer_pdb = YM_COMPLEX_overlap_homomerdimer_pdb.with_columns(
    comb_fold_submission = pl.col("Identifiers (and stoichiometry) of molecules in complex")
    .str.split("|")                                 
    .list.filter(~pl.element().str.starts_with("CHEBI:"))  
    .list.join(",")                                  
)

In [ ]:
YM_COMPLEX_overlap_homomerdimer_pdb.write_csv(data_dir/ "Pipeline/all_pdb_present_first_setup_pipeline_complexes.csv")

## Assemble iptm scores

In [ ]:
import polars as pl

In [ ]:
download_manifest_17_06_26 = pl.read_csv("/cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/afdb_pdbs_17_06_26/download_manifest.csv")

download_manifest_01_04_266 = pl.read_csv("/cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/afdb_pdbs_01_04_26/download_manifest.csv")

In [ ]:
merged_download_manifest = (
    pl.concat([download_manifest_17_06_26, download_manifest_01_04_266])
    .unique(subset="uniprotAccession", keep="first")
)

In [ ]:
merged_download_manifest["uniprotAccession","ipTM"].write_csv("/cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/merged_pdbs/homodimer_iptm.csv")

add heterodimer iptm scores

get all yeast pairs from YM and complex combinations

In [ ]:
#get pair between all yeast proteins
yeast_MAP_complexes_wConfidenceScores_wGenenames_total779_20251214 = pl.read_csv(
    data_dir_YM / "yeast.MAP_complexes_wConfidenceScores_wGenenames_total779_20251214.csv", separator = ",", has_header = True)

complex_db_entries = pl.read_csv(data_dir / "Complex_Portal/Saccharomyces_cerevisiae_ComplexTab.tsv", separator= "\t" )

In [ ]:
unique_ym_prot = (
    yeast_MAP_complexes_wConfidenceScores_wGenenames_total779_20251214
    .select(pl.col("UniProt_ACCs").str.split(" "))
    .explode("UniProt_ACCs")
    .get_column("UniProt_ACCs")
    .unique()
    .to_list()
)


col = "Identifiers (and stoichiometry) of molecules in complex"

unique_complex_db_entries = (
    complex_db_entries
    .select(
        # 1) remove brackets [ ]
        pl.col(col).str.replace_all(r"[\[\]]", "")
        # 2) Convert all pipes '|' into commas ',' 
        .str.replace_all(r"\|", ",")
        # 3) Now we can safely split on a literal comma!
        .str.split(",")
    )
    .explode(col)
    # 4) remove stoichiometry (x)
    .with_columns(
        pl.col(col)
        .str.replace_all(r"\(\d+\)", "")
        .str.strip_chars()
    )
    # 5) drop CHEBI entries and empty records
    .filter(~pl.col(col).str.starts_with("CHEBI:"))
    .filter(pl.col(col) != "")
    .get_column(col)
    .unique()
    .to_list()
)

In [23]:
all_yeast_prot = list(set(unique_complex_db_entries).union(unique_ym_prot))
#generate all possible pairs
protein_pairs = list(itertools.combinations(all_yeast_prot, 2))


In [25]:
pairs_df = pl.DataFrame(
    protein_pairs, 
    schema=["protein_A", "protein_B"],
    orient="row"
)


pairs_df.write_parquet(data_dir/"iPTM/yeast_protein_pairs.parquet")

In [27]:
pairs_df.head()

protein_A,protein_B
str,str
"""Q12754""","""P0CE41"""
"""Q12754""","""P04806"""
"""Q12754""","""P40164"""
"""Q12754""","""P25567"""
"""Q12754""","""CPX-1641"""


In [ ]:
homodimer_iptm = pl.read_csv(data_dir/"Pipeline/merged_pdbs/homodimer_iptm.csv")
heterodimer_iptm = pl.read_csv(data_dir/"Pipeline/iptm_scores.csv")

other stuff

In [ ]:
yeastmap_complex_pairs_with_scores = pl.read_csv("./Dataframes/Yeast_Map/yeastmap_complex_pairs_with_scores_incl_db.csv")
complex_by_breaking_values = pl.read_csv("./Dataframes/Yeast_Map/complex_by_breaking_values.csv")
YM_pred = pl.read_csv(data_dir_YM / "yeast.MAP_complexes_wConfidenceScores_wGenenames_total779_20251214.csv")
complex_db_entries = pl.read_csv(data_dir / "Complex_Portal/Saccharomyces cerevisiae_ComplexTab.tsv", separator= "\t" )


In [ ]:
YM_pred_with_index = (
    YM_pred.with_row_index("predicted_complex_id", offset = 1)
    .with_columns(
        pl.format("CPX_{}", pl.col("predicted_complex_id")).alias("predicted_complex_id"))
    .rename({"UniProt_ACCs": "predicted_complex"})
)

shoudk have 168 complexes with high confidence scores, for how many of them do I have a complex on COMPLEX (so complete overlap) for how many of the, do i have a stoichioemtry, for how many of them do I have a structure? what is theor iptm score

In [ ]:
pred_high_confidence_list = (YM_pred_with_index.filter(pl.col("ComplexConfidence") == 1).get_column("predicted_complex_id").unique().to_list())

In [ ]:
high_confidence_complex_by_breaking_values = complex_by_breaking_values.filter(pl.col("predicted_complex_id").is_in(pred_high_confidence_list))

In [ ]:
high_confidence_complex_by_breaking_values.height

## Do i have the nexessary AF data to run CF?

In [ ]:
pairs = pl.read_parquet(
    get_data_dir() / "25.12_pooled-ppi-yeast/data-26.03/summary_pairs.parquet"
) #takes around 1 min to load, as confirmed by jürgen the files from 26.04 also do dont contain the self pairs


In [ ]:
self_pairs = pairs.filter(pl.col("af3_id1") == pl.col("af3_id2"))
print(f"Total self-pairs: {self_pairs.height}")
